<a href="https://colab.research.google.com/github/normala127/NLP_Yelp_Review_Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics Project
## Predicting Restaurant Failure: An NLP Driven Risk Assesment from Yelp Reviews

Students: Hatidza Imamovic, Asja Basovic

### 1. Dataset creation and environment setup

Three datasets are needed to create the final dataset which will be used for analysis and model training. These are:
- business.json: holds data about each restaurant
- review.json: holds all the reviews for each restaurant
- checkin.json: holds the dates of all checked in visits in a given restaurant

Each is loaded and then combined in regards to the business_id to ensure a correct join. The final output is saved as final.json.


In [1]:
!pip install pyspark
!apt-get install openjdk-17-jdk-headless -qq


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("project").getOrCreate()

print(spark)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def show_shape(df):
  print((df.count(), len(df.columns)))

Loading the first dataset: business.json

In [6]:
df_business = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_business.json")
df_business.printSchema()
df_business.show()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

Loading the second dataset: review.json

In [7]:
df_review = spark.read.json(r"/content/drive/MyDrive/yelp_academic_dataset_review.json")

In [8]:
show_shape(df_business)

(150346, 14)


In [9]:
show_shape(df_review)

(6990280, 9)


Joining df_review and df_business into joined_df

In [10]:
df_review.createOrReplaceTempView("review")
df_business.createOrReplaceTempView("business")

In [11]:
joined_df = spark.sql("""
SELECT t1.*, t2.*
FROM review t1
LEFT JOIN business t2 ON t2.business_id = t1.business_id
""")
joined_df.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------------+-----+-----+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|         business_id|          categories|        city|               hours|is_open|     latitude|  longitude|                name|postal_code|review_count|stars|state|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------

In [12]:
joined_df = joined_df.drop(df_business['business_id'])

In [13]:
joined_df.columns # za obrisati

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars',
 'state']

In [14]:
show_shape(joined_df)

(6990280, 22)


Loading the third dataset: checkin.json

In [15]:
df_checkin = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_checkin.json")
df_checkin.printSchema()
df_checkin.show()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

+--------------------+--------------------+
|         business_id|                date|
+--------------------+--------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:...|
|--0iUa4sNDFiZFrAd...|2010-09-13 21:43:...|
|--30_8IhuyMHbSOcN...|2013-06-14 23:29:...|
|--7PUidqRWpRSpXeb...|2011-02-15 17:12:...|
|--7jw19RH9JKXgFoh...|2014-04-21 20:42:...|
|--8IbOsAAxjKRoYsB...|2015-06-06 01:03:...|
|--9osgUCSDUWUkoTL...|2015-06-13 02:00:...|
|--ARBQr1WMsTWiwOK...|2014-12-12 00:44:...|
|--FWWsIwxRwuw9vIM...|2010-09-11 16:28:...|
|--FcbSxK1AoEtEAxO...|2017-08-18 19:43:...|
|--LC8cIrALInl2vyo...|2017-01-12 19:10:...|
|--MbOh2O1pATkXa7x...|2013-04-21 01:52:...|
|--N9yp3ZWqQIm7DqK...|2012-10-06 20:46:...|
|--O3ip9NpXTKD4oBS...|2010-04-17 21:07:...|
|--OS_I7dnABrXvRCC...| 2018-05-11 18:23:36|
|--S43ruInmIsGrnnk...|2010-08-29 01:17:...|
|--SJXpAa0E-GCp2sm...|2014-04-06 22:23:...|
|--Sd93OFWITqDHifM...|2013-01-09 17

In [16]:
df_checkin_new=df_checkin.withColumnRenamed('date', 'date_checkin')
df_checkin_new.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date_checkin: string (nullable = true)



Joining joined_df (review+business) and df_checkin into joined_df2

In [17]:
df_checkin_new.createOrReplaceTempView('checkin')
joined_df.createOrReplaceTempView('joined_df')

In [18]:
joined_df2 = spark.sql("""
SELECT t1.*, t2.date_checkin
FROM joined_df t1
JOIN checkin t2 ON t2.business_id = t1.business_id
""")
joined_df2.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [19]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

Checking the distribution of opened and closed retaurants

In [20]:
df_isOpen=joined_df2.filter(joined_df2['is_open']==1)
df_isOpen.show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
df_isClosed=joined_df2.filter(joined_df2['is_open']==0)
df_isClosed.show()

In [ ]:
show_shape(df_isOpen) # 80% is open


In [ ]:
show_shape(df_isClosed) # 20% is closed

Dropping a part of open restaurants based on business_id to balance out the classes

In [21]:
from pyspark.sql.functions import col, hash

# id-level labels
id_labels = joined_df2.select("business_id", "is_open").distinct()

majority_ids = id_labels.filter(col("is_open") == 1) \
    .withColumn("keep", (hash("business_id") % 10) < 2)  # keep 20%

minority_ids = id_labels.filter(col("is_open") == 0) \
    .withColumn("keep", col("is_open").isNotNull())  # keep all

ids_to_keep = majority_ids.filter("keep").union(
    minority_ids.select("business_id", 'is_open', 'keep')
)

balanced_df = joined_df2.join(ids_to_keep.select("business_id"),
                      "business_id")

In [ ]:
show_shape(balanced_df)

In [ ]:
balanced_df.show()

In [ ]:
show_shape(majority_ids)

In [ ]:
show_shape(minority_ids)

Renaming columns and dropping uneeded columns

In [ ]:
balanced_df.printSchema()

In [22]:
cols=balanced_df.columns
stars_columns=[i for i, c in enumerate(cols) if c=='stars']

cols[stars_columns[0]]='stars_review'
cols[stars_columns[1]]='stars_business'

balanced_df=balanced_df.toDF(*cols)

In [23]:
balanced_df.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars_review',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars_business',
 'state',
 'date_checkin']

In [24]:
final_df=balanced_df.drop(*['cool', 'funny', 'useful', 'latitude', 'longitude', 'postal_code', 'attributes', 'hours'])

In [ ]:
final_df.printSchema()

In [ ]:
show_shape(final_df)

Saving the final dataset to the drive as a json file

In [ ]:
#final_df.write.mode("overwrite").parquet("/content/drive/MyDrive/dataprj/dataset2/dataset.parquet")

In [ ]:
#final_df.write.mode("overwrite").parquet("/content/test_parquet")

### 2. Preprocessing

Firstly, on a global level, null and duplicate values were dropped.

This section covers:
- lowercase,
- keep only letters (from all languages) and spaces,
- remove extra spaces,
- remove private information,
- emojis, urls.

It creates a pipeline for TF-IDF and for semantic analysis as slightly different cleaning techniques are used for each.

The section also uses n-grams, and handles "not" negation effectively.


In [59]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from pyspark.sql.functions import when, col, count, sum
from pyspark.sql import functions as F

from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, NGram, VectorAssembler
from pyspark.ml.functions import vector_to_array

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [26]:
df = final_df.select("*")

In [29]:
null_counts = df.select([count(when(col(c).isNull(), c).alias(c)) for c in df.columns])

null_counts.show()

+--------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------+
|c

In [30]:
df_nulls = df.filter(df['categories'].isNull())
df_nulls.limit(10)

DataFrame[business_id: string, date: string, review_id: string, stars_review: double, text: string, user_id: string, address: string, categories: string, city: string, is_open: bigint, name: string, review_count: bigint, stars_business: double, state: string, date_checkin: string]

In [ ]:
# TODO visaulaization

In [31]:
df = df.fillna({'categories': 'Unknown'})

In [32]:
df = df.dropDuplicates(['text'])
show_shape(df)

(4542946, 15)


In [48]:
def clean_sentiment(df):
  return df.withColumn("text_cleaned",
          F.trim(
            F.regexp_replace(
              F.lower(F.col("text")),
              r"http\S+|www\S+|\S+@\S+", " "
            ),
          )
      ).withColumn("text_cleaned_sentiment", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))


In [46]:
def clean_tfidf(df):

    return df.withColumn("text_cleaned",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.lower(F.col("text")),
                    r"http\S+|www\S+|\S+@\S+", " "
                ),
                r'[^\p{L}\s]+', " "
            )
        )
    ).withColumn("text_cleaned", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))

In [47]:
df=clean_tfidf(df)
#df.show()

In [49]:
df=clean_sentiment(df)

In [ ]:
#todo partitioning

In [ ]:
#show_shape(train_df)

In [ ]:
#show_shape(test_df)

### 3. Feature engineering

In [50]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500, suffix = ""):

  tokenizer = RegexTokenizer(
      inputCol="text_cleaned"+suffix,
      outputCol="tokens_raw"+suffix,
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw"+suffix, outputCol="filtered_tokens"+suffix, stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens"+suffix, outputCol="bigrams"+suffix)

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens"+suffix,
        outputCol="uni_count_features"+suffix,
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams'+suffix,
      outputCol = "bi_count_features"+suffix,
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features"+suffix, "bi_count_features"+suffix],
        outputCol="combined_counts"+suffix
    )

  idf = IDF(inputCol="combined_counts"+suffix, outputCol="features"+suffix, minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [56]:
tfidf_pipeline_stages = tfidf()
tfidf_pipeline = Pipeline(stages=tfidf_pipeline_stages)

In [52]:
df = df.withColumn('stars_review_binary', when(df['stars_review']<3, 0).otherwise(1))

In [53]:
def sentiment_analysis():
  stages = tfidf(suffix="_sentiment")

  log_reg = LogisticRegression(featuresCol='features_sentiment',
                             labelCol='stars_review_binary',  predictionCol='prediction_sentiment', probabilityCol = 'probability_sentiment', regParam=0.3, maxIter=10)

  stages.append(log_reg)

  return stages

In [57]:
sentiment_analysis_pipeline_stages = sentiment_analysis()
sentiment_analysis_pipeline = Pipeline(stages=sentiment_analysis_pipeline_stages)

In [58]:
unique_bus_df = df.select("business_id", "is_open").distinct()

fractions = {0: 0.8, 1: 0.8} # 80% of closed (0) and 80% of open (1)

train_ids = unique_bus_df.sampleBy("is_open", fractions, seed=42)

test_ids = unique_bus_df.join(train_ids, on="business_id", how="left_anti")

train_df = df.join(train_ids.select("business_id"), on="business_id", how="inner")
test_df = df.join(test_ids.select("business_id"), on="business_id", how="inner")

In [63]:
sentiment_model= sentiment_analysis_pipeline.fit(train_df)
tfidf_model = tfidf_pipeline.fit(train_df)

df_with_sentiment = sentiment_model.transform(df)

df_clean = df_with_sentiment.withColumn("sentiment_score", vector_to_array(F.col("probability_sentiment"))[1]) \
                            .drop("tokens_raw_sentiment", "filtered_tokens_sentiment", "uni_count_features_sentiment", "bi_count_features_sentiment", "combined_counts_sentiment","features_sentiment", "probability_sentiment")

final_feature_df = tfidf_model.transform(df_clean)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver

KeyboardInterrupt: 

Clean other columns
EDA
FE
Build and hypertune models (check literature)
Metrics and eval

In [65]:

smoke_test_df = df.sample(False, 0.001, seed=42).limit(1000).cache()
smoke_test_df.count() # This forces the cache to load

train_smoke, test_smoke = smoke_test_df.randomSplit([0.8, 0.2])


test_sentiment_model = sentiment_analysis_pipeline.fit(train_smoke)

test_output = test_sentiment_model.transform(smoke_test_df)

test_clean = test_output.withColumn("sentiment_score", vector_to_array(F.col("probability_sentiment"))[1]) \
                        .drop("tokens_raw_sentiment", "filtered_tokens_sentiment", "uni_count_features_sentiment",
                              "bi_count_features_sentiment", "combined_counts_sentiment", "features_sentiment",
                              "probability_sentiment")

final_smoke_check = tfidf_model.transform(test_clean)

final_smoke_check.show(5)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 